## **Assignment Simple machine Learning Project cycle**

### **_House Price Prediction Using California Housing Data_**
This dataset provides information about house prices in California. House Price Prediction


**In this notebook, you'll follow the basic machine learning process to build a regression model to predict house prices using the "California Housing Dataset" from sklearn.**



---



### **Follow the instructions and complete each TODO to complete the assessment on the essential steps in building and evaluating a regression model.**


The following is a description of each column in the dataset:

Dataset Features (California Housing):
* MedInc: Median income in block group
* HouseAge: Median house age in block group
* AveRooms: Average number of rooms per household
* AveBedrms: Average number of bedrooms per household
* Population: Block group population
* AveOccup: Average number of household members
* Latitude: Block group latitude
* Longitude: Block group longitude
* MedHouseVal (Target): Median house value in block group




### **Imports**

In [ ]:
# --- Imports ---
# TODO: Import all the necessary libraries for data handling, visualization, and model building.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

### **Data Collection or Loading**

In [ ]:
# --- Data Collection and Loading ---
# TODO: Load the 'California Housing' dataset from sklearn and convert it into a pandas DataFrame.

housing = fetch_california_housing()

data = pd.DataFrame(housing.data, columns=housing.feature_names)
data["MedHouseVal"] = housing.target   # add the target column

print(data.shape)

In [ ]:
# --- Quick Check of Data ---
# TODO: Display the first few rows of the dataset to understand its structure.

print(data.head())

In [ ]:
# TODO: Check the features and target variable. Identify which is continuous and categorical if applicable.

data.info()

In [ ]:
print(data.describe())

**Continuous or categorical?**

All 9 columns are numeric (`float64`) and **continuous**: MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude and the target MedHouseVal. There are no categorical columns in this dataset.

Because the target (MedHouseVal) is a continuous number, this is a **regression** problem. MedHouseVal is in units of $100,000 (for example 4.5 = $450,000), and MedInc is in units of $10,000.

### **EDA and Data Preprocessing**

In [ ]:
# --- EDA and Data Preprocessing ---
# TODO: Check for missing/null values.

print(data.isnull().sum())

In [ ]:
# TODO: Visualize the data. Create scatter plots to see the relationship between independent features and the target variable (MedHouseVal).
# Relationship between 'MedInc' and 'MedHouseVal'

plt.figure(figsize=(8, 5))
plt.scatter(data["MedInc"], data["MedHouseVal"], alpha=0.3)
plt.title("MedInc vs MedHouseVal")
plt.xlabel("MedInc (Median income)")
plt.ylabel("MedHouseVal (Median house value)")
plt.show()

In [ ]:
# TODO: Create a function to automate scatter plots for all features vs MedHouseVal.

def visualizeHousingData(data, features, target):
  """
  Creates a scatter plot of each feature against the target variable.

  Args:
    data: The pandas DataFrame containing the housing data.
    features: A list of column names to plot (e.g., ["MedInc", "AveRooms"]).
    target: The name of the target column (e.g., "MedHouseVal").

  Returns:
    None. Displays one scatter plot per feature.
  """

  for feature in features:
    plt.figure(figsize=(8, 5))
    plt.scatter(data[feature], data[target], alpha=0.3)
    plt.title(f"{feature} vs {target}")
    plt.xlabel(feature)
    plt.ylabel(target)
    plt.show()

In [ ]:
# TODO: Use the function to visualize the relationships between multiple features and the target variable.

features = ["MedInc", "AveRooms", "AveOccup", "HouseAge"]
target = "MedHouseVal"

visualizeHousingData(data, features, target)

In [ ]:
# Let's have a look at the correlation of all the columns with the MedHouseVal column:

correlation = data.corr()

print(correlation["MedHouseVal"].sort_values(ascending=False))

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix - California Housing")
plt.show()

**Observations from the EDA**

- There are **no missing values**, so no cleaning is needed.
- **MedInc** has the strongest positive relationship with house value (correlation about 0.69): areas with higher income have more expensive houses.
- **AveRooms, AveOccup and HouseAge** have a weak relationship with house value. AveRooms and AveOccup have some extreme outliers (very large values) that compress the plots.
- In all the plots, many points form a horizontal line at **MedHouseVal = 5**. This is because house values in the dataset were capped at $500,000.

### **ML Model Training**

In [ ]:
# --- ML Model Training ---
# TODO: Split the dataset into training and testing sets (80/20 split).

x = np.array(data.drop(columns=["MedHouseVal"]))
y = np.array(data["MedHouseVal"])
xtrain, xtest, ytrain, ytest = train_test_split(x, y,
                                                test_size=0.2,
                                                random_state=42)

print(xtrain.shape, xtest.shape)

In [ ]:
# TODO: Choose an appropriate regression model.

model = LinearRegression()

In [ ]:
# TODO: Train the model on the training data.

model.fit(xtrain, ytrain)

### **Model Evaluation**

In [ ]:
# --- Model Evaluation ---
# TODO: Evaluate the performance of the model on the test set using relevant metrics (e.g., RMSE, R-squared).

ypred = model.predict(xtest)

mse = mean_squared_error(ytest, ypred)
rmse = np.sqrt(mse)
r2 = r2_score(ytest, ypred)

print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R-squared: {r2}")

In [ ]:
# Actual vs Predicted values
plt.figure(figsize=(8, 5))
plt.scatter(ytest, ypred, alpha=0.3)
plt.plot([0, 5], [0, 5], color="red")   # perfect prediction line
plt.title("Actual vs Predicted MedHouseVal")
plt.xlabel("Actual MedHouseVal")
plt.ylabel("Predicted MedHouseVal")
plt.show()

**Interpretation of the results**

- **R-squared ≈ 0.58:** the model explains about 58% of the variation in house prices. This is a moderate result.
- **RMSE ≈ 0.75:** on average the predictions are wrong by about 0.75, which is about **$75,000**.
- In the Actual vs Predicted plot, the points are spread around the red line and the model predicts values above 5 even though real values are capped at 5.

The model is useful as a first version, but it is not very precise. Linear regression assumes straight-line relationships, and house prices depend strongly on location (latitude/longitude), which is not linear. A more advanced model (for example a Random Forest) or more feature engineering could improve the results.

### **Model Prediction**

In [ ]:
# --- Model Prediction ---
# TODO: Predict Median House Value from a new set of feature inputs.
# The order must be the same as the columns used for training:
# [MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude]

print(data.drop(columns=["MedHouseVal"]).columns.tolist())

features = np.array([[3, 30, 6, 1, 1500, 3, 34, -118]])
prediction = model.predict(features)

print(f"Predicted MedHouseVal: {prediction[0]}")
print(f"Predicted house value: ${prediction[0] * 100000:,.2f}")